In [43]:
import pandas as pd

df = pd.read_json('../../data/test/benchmark_zs_vs_ft.jsonl',orient='records',lines=True)

In [44]:
df.loc[df["quant"].str.startswith("Q8", na=False), "quant"] = "8bit"
df.loc[df["quant"].str.startswith("Q4", na=False), "quant"] = "4bit"


In [45]:
import re


def clean_model(name: str) -> str:
    s = str(name)

    # 1) rimuovo la parte di quantizzazione: _q4_..., _q8_...
    s = re.sub(r'_q[48]_.+', '', s, flags=re.IGNORECASE)

    # 2) rimuovo eventuali suffissi finali come _it, _instruct
    s = re.sub(r'_(it|instruct)$', '', s, flags=re.IGNORECASE)

    return s
manual_map = {
    # come da tue indicazioni
    "mamba_codestral_7b": "codestral_7b",
    "llama2": "codellama_7b",
    "deepseek": "deepseek_coder_6_7b",
    "qwen": "qwen2_5_coder_7b",
    "gemma":"codegemma_7b"
    # eventuali altri casi speciali puoi aggiungerli qui
    # "qualcosa": "nome_canonico",
}

def canonical_model(name: str) -> str:
    base = clean_model(name)
    return manual_map.get(base, base)  # se non è in manual_map, lascia base

df["model"] = df["model"].apply(canonical_model)


In [52]:
df[df['model']=='codestral_7b'].groupby(['model','variant','quant']).mean(['codebleu', 'codebert_f1',
       'bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL'])

row_id  codebleu  codebert_f1      bleu    meteor  \
model        variant quant                                                      
codestral_7b base    4bit    175.0  0.219645     0.751139  0.010446  0.086751   
                     8bit    175.0  0.214416     0.748367  0.010185  0.083161   

                              rouge1    rouge2    rougeL  
model        variant quant                                
codestral_7b base    4bit   0.280283  0.140466  0.236267  
                     8bit   0.280477  0.142906  0.241479

In [11]:
df.columns

Index(['model', 'variant', 'quant', 'row_id', 'codebleu', 'codebert_f1',
       'bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'generated', 'prompt',
       'reference', 'model_aligned'],
      dtype='object')